# FIFA World Cup 2026 — Match Prediction Pipeline

This notebook contains the full pipeline for predicting FIFA World Cup 2026 match outcomes using a Poisson regression model.

**Pipeline stages:**
1. Data Ingestion
2. Data Cleaning
3. Feature Engineering — Elo Ratings
4. Feature Engineering — Match Metadata
5. Model Training (Poisson GLM)
6. Tournament Simulation (Monte Carlo)

**Data required** (place in `../data/` relative to this notebook):
- `raw/results_1872_2026.csv` — international football results
- `raw/wc_2026_48_teams_fifa_rank_change_corrected.csv` — FIFA rankings
- `reference/group_stages.csv` — 2026 WC group assignments
- `reference/fixtures_knockout_wc2026.csv` — knockout bracket
- `reference/FIFA_confederations.csv` — confederation mappings


---
# Section 1 — Data Ingestion

Load all raw data files and inspect their contents before any cleaning or transformation.

In [ ]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data")

df_games = pd.read_csv(DATA_DIR / 'raw/results_1872_2026.csv')
df_ranking = pd.read_csv(DATA_DIR / 'raw/wc_2026_48_teams_fifa_rank_change_corrected.csv')
df_group_stage = pd.read_csv(DATA_DIR / 'reference/group_stages.csv', sep=';')
df_knockout_fixtures = pd.read_csv(DATA_DIR / 'reference/fixtures_knockout_wc2026.csv')

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

# All Games since 1872 - 2026

In [ ]:
print(df_games.shape)
df_games.head()

In [ ]:
# Last 72 rows are all for the WC 2026
df_games.tail()

# FIFA Ranking in 2022 and 2026

In [ ]:
print(df_ranking.shape)
df_ranking.head()

In [ ]:
df_ranking['Nation'].unique()

# Group Stages

In [ ]:
print(df_group_stage.shape)
df_group_stage.head()

# Knowckout Fixtures Grid

In [ ]:
print(df_knockout_fixtures.shape)
df_knockout_fixtures.head()

---
# Section 2 — Data Cleaning

Split the raw dataset into:
- **Historical matches** (49,215 rows): completed matches with known scores — used for training
- **WC 2026 fixtures** (72 rows): upcoming group-stage matches with no scores — used for prediction

Save both to `../data/interim/`.

In [ ]:
print(df_games.shape)
df_games.head(2)

In [ ]:
df_games.tail(2)

In [ ]:
# Split into historical matches and WC2026 fixtures
historical_matches = df_games[df_games['home_score'].notna()].copy()
wc2026_fixtures = df_games[df_games['home_score'].isna()].copy()

In [ ]:
print(historical_matches.shape)
historical_matches.head(1)

In [ ]:
print(wc2026_fixtures.shape)
wc2026_fixtures.head(1)

In [ ]:
# Save to CSV
historical_matches.to_csv(DATA_DIR / 'interim/historical_matches.csv', index=False)
wc2026_fixtures.to_csv(DATA_DIR / 'interim/wc2026_fixtures.csv', index=False)

print(f"Historical matches: {len(historical_matches)}")
print(f"WC2026 fixtures: {len(wc2026_fixtures)}")

---
# Section 3 — Feature Engineering: Elo Ratings

Compute Elo ratings for every international team across all historical matches.

**Design decisions:**
- K-factor varies by tournament importance (60 for World Cup, 20 for friendlies)
- Home advantage = 100 Elo points; set to 0 for neutral-venue matches
- Goal-difference multiplier: bigger wins produce larger Elo shifts
- CONIFA (non-FIFA) tournaments are excluded

The key output is `df_match_features.csv` — a table where every historical match has pre-match Elo ratings for both teams.


In [ ]:
print(df_games.shape)
df_games.head(2)

In [ ]:
df_games['tournament'].value_counts().head()

In [ ]:
# Exclude non-FIFA tournaments / Matches

df_games = df_games[~df_games['tournament'].str.startswith('CONIFA')]
df_games = df_games[~df_games['tournament'].str.startswith('ConIFA')]

print(df_games.shape)

# Compute Elo ratings for every international team over history

In [ ]:
import pandas as pd
from collections import defaultdict

# ============================================================
# Compute Elo ratings for every international team over history
# ============================================================

# ---------- 1. K-factor tiers by tournament importance ----------
TIER_1_WORLD_CUP = {'FIFA World Cup'}

TIER_2_CONTINENTAL = {
    'UEFA Euro', 'Copa América', 'African Cup of Nations', 'AFC Asian Cup',
    'Gold Cup', 'CONCACAF Championship', 'Oceania Nations Cup',
    'Confederations Cup',
}

TIER_3_QUALIFIERS_NATIONS = {
    'FIFA World Cup qualification', 'UEFA Euro qualification',
    'African Cup of Nations qualification', 'AFC Asian Cup qualification',
    'Gold Cup qualification', 'CONCACAF Championship qualification',
    'Copa América qualification', 'Oceania Nations Cup qualification',
    'UEFA Nations League', 'CONCACAF Nations League',
    'CONCACAF Nations League qualification',
}

TIER_4_REGIONAL = {
    # Africa
    'CECAFA Cup', 'COSAFA Cup', 'COSAFA Cup qualification', 'WAFF Championship',
    'Amílcar Cabral Cup', 'All-African Games', 'UDEAC Cup', 'UNIFFAC Cup',
    'West African Cup', 'Nile Basin Tournament', 'African Friendship Games',
    # Asia / Oceania
    'Gulf Cup', 'Arab Cup', 'Arab Cup qualification', 'SAFF Cup',
    'AFF Championship', 'AFF Championship qualification', 'EAFF Championship',
    'EAFF Championship qualification', 'ASEAN Championship',
    'ASEAN Championship qualification', 'AFC Challenge Cup',
    'AFC Challenge Cup qualification', 'Asian Games', 'CAFA Nations Cup',
    'Southeast Asian Games', 'South Asian Games', 'Dynasty Cup',
    'Pacific Games', 'South Pacific Games', 'Melanesia Cup',
    'Indian Ocean Island Games', 'Afro-Asian Games',
    # Europe
    'British Home Championship', 'Nordic Championship', 'Baltic Cup',
    'Balkan Cup', 'Central European International Cup',
    # Americas
    'CFU Caribbean Cup', 'CFU Caribbean Cup qualification', 'UNCAF Cup',
    'Central American and Caribbean Games', 'Pan American Championship',
    'CCCF Championship', 'Bolivarian Games', 'NAFC Championship',
    # Multi-sport
    'Olympic Games',
}

TIER_5_FRIENDLY = {'Friendly', 'FIFA Series', 'CONCACAF Series'}


def get_k_factor(tournament):
    """K controls how reactive Elo is to a match. Higher = bigger swings."""
    if tournament in TIER_1_WORLD_CUP:          return 60
    if tournament in TIER_2_CONTINENTAL:        return 50
    if tournament in TIER_3_QUALIFIERS_NATIONS: return 40
    if tournament in TIER_4_REGIONAL:           return 30
    if tournament in TIER_5_FRIENDLY:           return 20
    return 15   # minor/exhibition tournaments, anniversary cups, etc.


# ---------- 2. Expected score formula ----------
def expected_score(home_elo, away_elo, home_advantage=100):
    # home_advantage = 100 → ~64% win rate for evenly matched home teams.
    # Set to 0 for neutral-venue matches (the formula then becomes symmetric).
    diff = (home_elo + home_advantage) - away_elo
    return 1 / (1 + 10 ** (-diff / 400))


# ---------- 3. Load and prepare ----------
df = df_games.copy()
df = df.dropna(subset=['home_score', 'away_score'])

# Drop CONIFA tournaments — these only involve non-FIFA teams
df = df[~df['tournament'].str.contains('CONIFA', case=False, na=False)]

df = df.sort_values('date').reset_index(drop=True)
print(f"Computing Elo over {len(df):,} historical matches")

# ---------- 4. Iterate through history ----------
INITIAL_ELO = 1500
current_elo = defaultdict(lambda: INITIAL_ELO)
elo_history = []

for row in df.itertuples(index=False):
    home, away = row.home_team, row.away_team
    home_elo, away_elo = current_elo[home], current_elo[away]

    # Neutral venue cancels home advantage
    h_adv = 0 if row.neutral else 100
    exp_home = expected_score(home_elo, away_elo, h_adv)
    exp_away = 1 - exp_home

    # Actual result
    if row.home_score > row.away_score:
        actual_home, actual_away = 1.0, 0.0
    elif row.home_score < row.away_score:
        actual_home, actual_away = 0.0, 1.0
    else:
        actual_home, actual_away = 0.5, 0.5

    # Goal-difference multiplier — bigger wins move Elo more
    goal_diff = abs(row.home_score - row.away_score)
    if goal_diff <= 1:
        g = 1.0
    elif goal_diff == 2:
        g = 1.5
    else:
        g = (11 + goal_diff) / 8

    K = get_k_factor(row.tournament)

    # Update both teams (zero-sum: total Elo across the two teams is preserved)
    current_elo[home] = home_elo + K * g * (actual_home - exp_home)
    current_elo[away] = away_elo + K * g * (actual_away - exp_away)

    elo_history.append((row.date, home, current_elo[home]))
    elo_history.append((row.date, away, current_elo[away]))

# ---------- 5. Build the lookup table ----------
df_elo = pd.DataFrame(elo_history, columns=['date', 'team', 'elo_after'])
df_elo['date'] = pd.to_datetime(df_elo['date'])
print(f"df_elo shape: {df_elo.shape}")

# ---------- 6. Sanity checks ----------
print("\n--- Top 15 current Elo ---")
current_ranking = (
    df_elo.sort_values('date')
          .groupby('team')
          .tail(1)
          .sort_values('elo_after', ascending=False)
          .head(15)
)
print(current_ranking.to_string(index=False))

print("\n--- Bottom 15 current Elo (sanity: should be tiny non-FIFA or minnow teams) ---")
print(
    df_elo.sort_values('date')
          .groupby('team')
          .tail(1)
          .sort_values('elo_after')
          .head(15)
          .to_string(index=False)
)

print(f"\nTotal unique teams with Elo: {df_elo['team'].nunique()}")

### -> FIFA Ranking AND highly correlated but not identical — exactly what you want. 

##  neutral-flag distribution

In [ ]:
print("Neutral venue distribution:")
print(df_games['neutral'].value_counts(normalize=True))
print("\nNeutral by tournament (top 10):")
print(df_games.groupby('tournament')['neutral']
              .mean()
              .sort_values(ascending=False)
              .head())

## Quick checks

In [ ]:
# Spot check: Morocco should have spiked after WC2022
morocco = df_elo[df_elo['team'] == 'Morocco'].sort_values('date')
print(morocco.tail(20))   # look for jump around Dec 2022

# Spot check: a team that played few matches should be near 1500
print(df_elo[df_elo['team'] == 'Anguilla'].tail(5))

In [ ]:
df_games.head()

In [ ]:
df_elo.head()

# Elo impact on fixtures (with Leakage but removed later)

In [ ]:
# For each historical match, find each team's Elo state around that match.
# We'll grab BOTH the pre-match Elo (legitimate feature) and the post-match Elo
# (which we'll use to demonstrate target leakage, then drop before training).

df_games['date'] = pd.to_datetime(df_games['date'])
df_elo['date']   = pd.to_datetime(df_elo['date'])

df_elo_sorted   = df_elo.sort_values('date').reset_index(drop=True)
df_games_sorted = df_games.sort_values('date').reset_index(drop=True)

# ----- HOME: pre-match Elo (lookup strictly BEFORE the match date) -----
df_temp = pd.merge_asof(
    df_games_sorted,
    df_elo_sorted.rename(columns={'team': 'home_team', 'elo_after': 'home_elo_pre'}),
    on='date',
    by='home_team',
    direction='backward',
    allow_exact_matches=False,   # exclude same-day entries → no leakage
)

# ----- HOME: post-match Elo (lookup ON the match date — this is the leak) -----
df_temp = pd.merge_asof(
    df_temp.sort_values('date'),
    df_elo_sorted.rename(columns={'team': 'home_team', 'elo_after': 'home_elo_after'}),
    on='date',
    by='home_team',
    direction='backward',
    allow_exact_matches=True,    # INCLUDE same-day → grabs the post-match update
)

# ----- AWAY: pre-match Elo -----
df_temp = pd.merge_asof(
    df_temp.sort_values('date'),
    df_elo_sorted.rename(columns={'team': 'away_team', 'elo_after': 'away_elo_pre'}),
    on='date',
    by='away_team',
    direction='backward',
    allow_exact_matches=False,
)

# ----- AWAY: post-match Elo (the leak again) -----
df_match_features = pd.merge_asof(
    df_temp.sort_values('date'),
    df_elo_sorted.rename(columns={'team': 'away_team', 'elo_after': 'away_elo_after'}),
    on='date',
    by='away_team',
    direction='backward',
    allow_exact_matches=True,
)

df_match_features['elo_diff'] = (
    df_match_features['home_elo_pre'] - df_match_features['away_elo_pre']
)

# Inspect: pre vs after side by side
df_match_features[[
    'date', 'home_team', 'away_team', 'home_score', 'away_score',
    'home_elo_pre', 'home_elo_after', 'away_elo_pre', 'away_elo_after'
]].tail(10)

# Elo on fixtures

In [ ]:
# For each historical match, find each team's Elo *just before* that match.
# (You want pre-match Elo as the feature, not post-match — post-match would leak the result.)

df_games['date'] = pd.to_datetime(df_games['date'])
df_elo['date'] = pd.to_datetime(df_elo['date'])

df_elo_sorted = df_elo.sort_values('date').reset_index(drop=True)

# Use merge_asof for "find the most recent Elo entry before this match date"
df_games_sorted = df_games.sort_values('date').reset_index(drop=True)

# Home team Elo at time of match
df_with_home_elo = pd.merge_asof(
    df_games_sorted,
    df_elo_sorted.rename(columns={'team': 'home_team', 'elo_after': 'home_elo_pre'}),
    on='date',
    by='home_team',
    direction='backward',
    allow_exact_matches=False  # don't pick up the *current* match's post-Elo
)

# Same for away team
df_match_features = pd.merge_asof(
    df_with_home_elo.sort_values('date'),
    df_elo_sorted.rename(columns={'team': 'away_team', 'elo_after': 'away_elo_pre'}),
    on='date',
    by='away_team',
    direction='backward',
    allow_exact_matches=False
)

df_match_features['elo_diff'] = df_match_features['home_elo_pre'] - df_match_features['away_elo_pre']

In [ ]:
print(df_match_features.shape)
df_match_features.tail(2)

In [ ]:
assert 'home_elo_after' not in df_match_features.columns
assert 'away_elo_after' not in df_match_features.columns
assert 'home_elo_change' not in df_match_features.columns
print("✓ No leaky columns")

In [ ]:
df_match_features.to_csv(DATA_DIR / 'processed/df_match_features.csv', index=False)
print(f"Saved: {df_match_features.shape}")

---
# Section 4 — Feature Engineering: Match Metadata

Add tournament tier weights, confederation labels, recent form, and head-to-head records.

**Outputs saved to `../data/processed/`:**
- `df_match_features.csv` — full feature table (all historical matches)
- `df_form_2026.csv` — recent goal differential per WC team
- `df_h2h_2026.csv` — head-to-head record for WC team pairings


In [ ]:
import pandas as pd
df = pd.read_csv(DATA_DIR / 'processed/df_match_features.csv', parse_dates=['date'])
conf = pd.read_csv(DATA_DIR / 'reference/FIFA_confederations.csv')

print(df.shape)
df.head(3)

## Section 0 — Load and handle Elo NaN

In [ ]:
# Fill cold-start Elo NaNs with starting value
df['home_elo_pre'] = df['home_elo_pre'].fillna(1500)
df['away_elo_pre'] = df['away_elo_pre'].fillna(1500)
df['elo_diff'] = df['home_elo_pre'] - df['away_elo_pre']

print(df.shape)
print("Remaining NaN in Elo:", df[['home_elo_pre', 'away_elo_pre']].isna().sum().sum())

In [ ]:
df.head(1)

## Section 1 — Tournament TIERS + tournament_weight

In [ ]:
# Tournament tier sets — same buckets used for Elo K-factor in Notebook 2.
# Re-defined here because notebooks don't share state.

TIER_1_WORLD_CUP = {'FIFA World Cup'}

TIER_2_CONTINENTAL = {
    'UEFA Euro', 'Copa América', 'African Cup of Nations', 'AFC Asian Cup',
    'Gold Cup', 'CONCACAF Championship', 'Oceania Nations Cup',
    'Confederations Cup',
}

TIER_3_QUALIFIERS_NATIONS = {
    'FIFA World Cup qualification', 'UEFA Euro qualification',
    'African Cup of Nations qualification', 'AFC Asian Cup qualification',
    'Gold Cup qualification', 'CONCACAF Championship qualification',
    'Copa América qualification', 'Oceania Nations Cup qualification',
    'UEFA Nations League', 'CONCACAF Nations League',
    'CONCACAF Nations League qualification',
}

TIER_4_REGIONAL = {
    # Africa
    'CECAFA Cup', 'COSAFA Cup', 'COSAFA Cup qualification', 'WAFF Championship',
    'Amílcar Cabral Cup', 'All-African Games', 'UDEAC Cup', 'UNIFFAC Cup',
    'West African Cup', 'Nile Basin Tournament', 'African Friendship Games',
    # Asia / Oceania
    'Gulf Cup', 'Arab Cup', 'Arab Cup qualification', 'SAFF Cup',
    'AFF Championship', 'AFF Championship qualification', 'EAFF Championship',
    'EAFF Championship qualification', 'ASEAN Championship',
    'ASEAN Championship qualification', 'AFC Challenge Cup',
    'AFC Challenge Cup qualification', 'Asian Games', 'CAFA Nations Cup',
    'Southeast Asian Games', 'South Asian Games', 'Dynasty Cup',
    'Pacific Games', 'South Pacific Games', 'Melanesia Cup',
    'Indian Ocean Island Games', 'Afro-Asian Games',
    # Europe
    'British Home Championship', 'Nordic Championship', 'Baltic Cup',
    'Balkan Cup', 'Central European International Cup',
    # Americas
    'CFU Caribbean Cup', 'CFU Caribbean Cup qualification', 'UNCAF Cup',
    'Central American and Caribbean Games', 'Pan American Championship',
    'CCCF Championship', 'Bolivarian Games', 'NAFC Championship',
    # Multi-sport
    'Olympic Games',
}

In [ ]:
def tournament_weight(t):
    if t in TIER_1_WORLD_CUP:          return 5
    if t in TIER_2_CONTINENTAL:        return 4
    if t in TIER_3_QUALIFIERS_NATIONS: return 3
    if t in TIER_4_REGIONAL:           return 2
    return 1   # friendlies + minor exhibitions

df['tournament_weight'] = df['tournament'].apply(tournament_weight)
print(df['tournament_weight'].value_counts().sort_index())

## Section 2 — is_competitive  

- If not friendly then competitive

In [ ]:
df['is_competitive'] = df['tournament'] != 'Friendly'
print(df['is_competitive'].value_counts())

## Section 3 — Confederation merge

In [ ]:
df = df.merge(
    conf.rename(columns={'nation': 'home_team', 'confederation': 'home_confederation'}),
    on='home_team', how='left'
)
df = df.merge(
    conf.rename(columns={'nation': 'away_team', 'confederation': 'away_confederation'}),
    on='away_team', how='left'
)

print("Missing home confederation:", df['home_confederation'].isna().sum())
print("Missing away confederation:", df['away_confederation'].isna().sum())

### Important note here:

your FIFA_confederations.csv only has the 48 qualified teams. So thousands of historical matches will get NaN for confederation — Czechoslovakia, Yugoslavia, Bolivia, Senegal in 1990, etc.

We fill empty values with 'unknown'

In [ ]:
df['home_confederation'] = df['home_confederation'].fillna('Unknown')
df['away_confederation'] = df['away_confederation'].fillna('Unknown')

In [ ]:
print("Missing home confederation:", df['home_confederation'].isna().sum())
print("Missing away confederation:", df['away_confederation'].isna().sum())

In [ ]:
df.head(2)

## Section 4 — Recent goal differential (last 30 matches before WC2026)

In [ ]:
# Last 30 competitive matches per team, before the World Cup
cutoff = '2026-06-01'

# Stack home + away into one view (still needed — Brazil is in two columns)
home_view = df[['date', 'home_team', 'home_score', 'away_score', 'tournament']].rename(
    columns={'home_team': 'team', 'home_score': 'gf', 'away_score': 'ga'})
away_view = df[['date', 'away_team', 'away_score', 'home_score', 'tournament']].rename(
    columns={'away_team': 'team', 'away_score': 'gf', 'home_score': 'ga'})
long = pd.concat([home_view, away_view], ignore_index=True)

# Filter: before WC + competitive only
long = long[(long['date'] < cutoff) & (long['tournament'] != 'Friendly')]
long['gd'] = long['gf'] - long['ga']

# For each team, take their last 30 matches and sum
form = (
    long.sort_values('date')
        .groupby('team')
        .tail(30)
        .groupby('team')['gd']
        .sum()
        .reset_index()
        .rename(columns={'gd': 'form_score'})
)

# Keep just the 48 WC teams
df_groups = pd.read_csv(DATA_DIR / 'reference/group_stages.csv', sep=';')
wc_teams = df_groups['nation'].unique()
form = form[form['team'].isin(wc_teams)].sort_values('form_score', ascending=False)

print(form)


## Section 5 — Head-to-head record (last 5 meetings)

In [ ]:
# ===== Section 5: Head-to-head record (last 5 meetings before WC2026) =====

cutoff = '2026-06-01'

# Filter to competitive matches before the World Cup
h2h_data = df[
    (df['date'] < cutoff) &
    (df['tournament'] != 'Friendly')
].copy()

In [ ]:
# Goal differential from home_team's perspective
h2h_data['gd'] = h2h_data['home_score'] - h2h_data['away_score']

# Direction-agnostic pair key: alphabetical so Argentina vs Brazil == Brazil vs Argentina
h2h_data['team_a'] = h2h_data[['home_team', 'away_team']].min(axis=1)
h2h_data['team_b'] = h2h_data[['home_team', 'away_team']].max(axis=1)

# Convert goal diff to "from team_a's perspective"
# If team_a was home, keep the sign; if team_a was away, flip it
h2h_data['gd_for_a'] = h2h_data.apply(
    lambda r: r['gd'] if r['home_team'] == r['team_a'] else -r['gd'],
    axis=1
)

In [ ]:
# For each pair, take their last 5 meetings, sum the goal diff, and count meetings
h2h = (
    h2h_data.sort_values('date')
            .groupby(['team_a', 'team_b'])
            .tail(5)
            .groupby(['team_a', 'team_b'])
            .agg(h2h_score=('gd_for_a', 'sum'),
                 n_meetings_analysed=('gd_for_a', 'size'))
            .reset_index()
)

In [ ]:
# Zero out h2h for low-sample pairs (avoids noise from 1-2 ancient meetings)
h2h.loc[h2h['n_meetings_analysed'] < 3, 'h2h_score'] = 0

In [ ]:
# Keep only pairings between WC2026 teams
wc_teams = set(df_groups['nation'])
h2h = h2h[h2h['team_a'].isin(wc_teams) & h2h['team_b'].isin(wc_teams)]

print(f"Pairings with history: {len(h2h)}")
print("\nTop 10 most lopsided rivalries:")
print(h2h.reindex(h2h['h2h_score'].abs().sort_values(ascending=False).index).head(10))

In [ ]:
#### Spot check one rivalry:
print(h2h[(h2h['team_a'] == 'Argentina') & (h2h['team_b'] == 'Brazil')])
print("------------------------------------------------------------------")
print(h2h[(h2h['team_a'] == 'Algeria') & (h2h['team_b'] == 'Morocco')])
print("------------------------------------------------------------------")
print(h2h[(h2h['team_a'] == 'France') & (h2h['team_b'] == 'Senegal')])
print("------------------------------------------------------------------")
print(h2h[(h2h['team_a'] == 'France') & (h2h['team_b'] == 'Spain')])

- Since Senegal and france have only played twice officially then we didn't do an h2h score for them

In [ ]:
df[
    (((df['home_team'] == 'France') & (df['away_team'] == 'Senegal')) |
     ((df['home_team'] == 'Senegal') & (df['away_team'] == 'France'))) &
    (df['tournament'] != 'Friendly')
][['date', 'home_team', 'away_team', 'home_score', 'away_score', 'tournament']]

## Section 6 — Final sanity checks + save

In [ ]:
# --- Sanity checks on df_match_features ---
print("df shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nNaN counts:")
nan_counts = df.isna().sum()
print(nan_counts[nan_counts > 0] if nan_counts.sum() > 0 else "✓ No NaN values")

In [ ]:
# --- Sanity checks on form table ---
print(f"\nform table: {form.shape[0]} teams (expected 48)")
assert form.shape[0] == 48, "Missing teams in form table"

In [ ]:
# --- Sanity checks on h2h table ---
print(f"h2h table: {h2h.shape[0]} pairings between WC2026 teams")

In [ ]:
# --- Save ---
df.to_csv(DATA_DIR / 'processed/df_match_features.csv', index=False)
form.to_csv(DATA_DIR / 'processed/df_form_2026.csv', index=False)
h2h.to_csv(DATA_DIR / 'processed/df_h2h_2026.csv', index=False)

print("\n✓ Saved all three files to data/processed/")

---
# Section 5 — Model Training

Train two independent **Poisson GLM** models:
- **Model A** — predicts expected home goals (λ_home)
- **Model B** — predicts expected away goals (λ_away)

**Features used (7):**
- `home_elo_pre`, `away_elo_pre`, `elo_diff` — team strength
- `tournament_weight` — match importance
- `neutral` — venue type
- `home_confederation`, `away_confederation` — one-hot encoded

**Training set:** competitive matches from 2000–2023  
**Validation set:** competitive matches from 2024–2026

**Note:** form, H2H, and FIFA rank are *not* training features — they are used as contextual adjustments in the simulation stage.


## This notebook uses df_match_features.csv only. 
## Form, h2h, and FIFA rank are applied as adjustments in the simulation notebook, not as training features.

#
---

In [ ]:
import pandas as pd
df = pd.read_csv(DATA_DIR / 'processed/df_match_features.csv', parse_dates=['date'])

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nDtypes:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum()[df.isna().sum() > 0])

### What the model will actually use
Out of those 16 columns, here's what's what:

##### Identifiers (not features): date, home_team, away_team, city, country. These tell you which match it is, not anything predictive.

##### Targets: home_score, away_score. The things you're predicting. They can't be in the features.
##### Drop: tournament, is_competitive. tournament has too many unique values to one-hot encode sensibly (200+ tournaments). is_competitive is information you'll use to filter the training set, not as a feature.

Actual features to feed the model (7):

- home_elo_pre — home team strength
- away_elo_pre — away team strength
- elo_diff — useful for simpler models, redundant for tree models but harmless
- tournament_weight — match importance
- neutral — venue type (boolean)
- home_confederation — categorical (one-hot it)
- away_confederation — categorical (one-hot it)

That's a focused, principled feature set. Not 50 columns of noise.

In [ ]:
print(df.shape)
df.head(2)

In [ ]:
df['is_competitive'].value_counts()   

In [ ]:
# Use only competitive matches for training (remove friendlies and the cold-start NaN rows)
train_df = df[df['is_competitive']].copy()
print(f"Training on {len(train_df):,} competitive matches")

### Why two separate Poisson models
This is worth understanding before you write code. Football models predict goals as two independent Poisson distributions:

- Model A: predict λ_home (expected goals scored by home team)
- Model B: predict λ_away (expected goals scored by away team)

Both models use the same features, just with the target swapped. Then you sample from each Poisson to get score lines, and outcomes derive from there.

#
----

# SECTION 2: Poisson Models

# Train test split

In [ ]:
# Build the training set / Filter out before 2000
train_df = df[df['is_competitive'] & (df['date'] >= '2000-01-01')].copy()

train = train_df[train_df['date'] < '2024-01-01']
valid = train_df[train_df['date'] >= '2024-01-01']

print(f"Train: {len(train):,} matches ({train['date'].min().year}–{train['date'].max().year})")
print(f"Valid: {len(valid):,} matches ({valid['date'].min().year}–{valid['date'].max().year})")

#### I picked 2000 to balance sample size against tactical era stability. Earlier would dilute the signal, later would limit data

# Fit 2 Poisson Models

In [ ]:
import statsmodels.api as sm
import pandas as pd

# Define the feature columns
numeric_features = ['home_elo_pre', 'away_elo_pre', 'tournament_weight']
categorical_features = ['home_confederation', 'away_confederation']

def build_design_matrix(df):
    """Convert features into a numeric matrix the model can consume."""
    X = df[numeric_features].copy()
    # Neutral venue as a boolean → int
    X['neutral'] = df['neutral'].astype(int)
    # One-hot encode confederations (drop_first avoids the dummy-variable trap)
    cats = pd.get_dummies(df[categorical_features], drop_first=True).astype(int)
    X = pd.concat([X, cats], axis=1)
    # Add intercept column (required by statsmodels GLM)
    X = sm.add_constant(X, has_constant='add')
    return X

# Build design matrices
X_train = build_design_matrix(train)
X_valid = build_design_matrix(valid)

# Make sure validation has the same columns as train (handles any missing categories)
X_valid = X_valid.reindex(columns=X_train.columns, fill_value=0)

# Targets
y_home_train = train['home_score']
y_away_train = train['away_score']

# Fit Model A: expected home goals
model_home = sm.GLM(y_home_train, X_train, family=sm.families.Poisson()).fit()

# Fit Model B: expected away goals
model_away = sm.GLM(y_away_train, X_train, family=sm.families.Poisson()).fit()

print("=== Home goals model ===")
print(model_home.summary())
print("\n=== Away goals model ===")
print(model_away.summary())

- That neutral: +0.37 for away goals is the cleanest validation that your model has correctly identified home advantage. 
- Without home support, the "away" team scores significantly more. That's a real, well-known effect in football.

### The single most important number to internalize
The two Elo coefficients (home and away) are your model's beating heart. They confirm:

- A team with 200 more Elo points than its opponent scores ~exp(0.0016×200) = 38% more goals while conceding ~exp(0.0021×200) = 52% fewer goals.
- For Argentina (Elo ~2180) vs Saudi Arabia (~1500): a 680-point gap → Argentina scores ~exp(0.0016×680) = 3x as many goals and concedes ~exp(0.0021×680) = ¼ as many. This is how the model knows Argentina should win most of those matchups.

#
-----

# Section 3 : Validation on holdout data (2024-2026)

## Check 1: Mean Absolute Error on goal predictions

In [ ]:
# Predict expected goals for validation set
y_home_pred = model_home.predict(X_valid)
y_away_pred = model_away.predict(X_valid)

# Actual goals
y_home_actual = valid['home_score']
y_away_actual = valid['away_score']

# Mean absolute error
mae_home = (y_home_pred - y_home_actual).abs().mean()
mae_away = (y_away_pred - y_away_actual).abs().mean()

print(f"Home goals MAE: model predicts HOME goals off by {mae_home:.3f}")
print(f"Away goals MAE: model predicts AWAY goals off by {mae_away:.3f}")
print(f"Mean predicted home goals: {y_home_pred.mean():.2f} (actual: {y_home_actual.mean():.2f})")
print(f"Mean predicted away goals: {y_away_pred.mean():.2f} (actual: {y_away_actual.mean():.2f})")

- My model overpredicts home goals by about 0.12 on average.
- It's small, but it tells me the model has slightly too much faith in home advantage. 
- For the World Cup, every match is neutral anyway — so home advantage doesn't apply, and this bias won't affect the simulation.

PS: The Poisson model is deterministic If you rerun model_home.predict(X_valid) ten times, you'll get exactly the same numbers every time. 

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Home goals
axes[0].scatter(y_home_pred, y_home_actual, alpha=0.15, s=15)
axes[0].plot([0, 6], [0, 6], 'r--', label='Perfect prediction')
axes[0].set_xlabel('Predicted home goals')
axes[0].set_ylabel('Actual home goals')
axes[0].set_title(f'Home goals (MAE = {mae_home:.2f})')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Away goals
axes[1].scatter(y_away_pred, y_away_actual, alpha=0.15, s=15, color='orange')
axes[1].plot([0, 6], [0, 6], 'r--', label='Perfect prediction')
axes[1].set_xlabel('Predicted away goals')
axes[1].set_ylabel('Actual away goals')
axes[1].set_title(f'Away goals (MAE = {mae_away:.2f})')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

- The cloud of dots clusters around the red diagonal — meaning when the model predicts ~2 goals, actual goals come out around 2 on average. 
- The vertical spread is the irreducible noise of football: you can predict the average, but never the exact score.

## Check 2: Outcome accuracy (Monte Carlo Simulation)

In [ ]:
import numpy as np

def predict_outcome(lambda_h, lambda_a, n_sims=10000):
    """Sample from both Poissons many times, return outcome probabilities."""
    home_goals = np.random.poisson(lambda_h, n_sims)
    away_goals = np.random.poisson(lambda_a, n_sims)
    p_home = (home_goals > away_goals).mean()
    p_draw = (home_goals == away_goals).mean()
    p_away = (home_goals < away_goals).mean()
    return p_home, p_draw, p_away

# Get predicted outcome for each validation match
valid = valid.reset_index(drop=True)
outcome_probs = [predict_outcome(h, a) for h, a in zip(y_home_pred, y_away_pred)]
valid['p_home'] = [o[0] for o in outcome_probs]
valid['p_draw'] = [o[1] for o in outcome_probs]
valid['p_away'] = [o[2] for o in outcome_probs]

# Predicted outcome = highest probability
valid['pred_outcome'] = valid[['p_home', 'p_draw', 'p_away']].idxmax(axis=1).str.replace('p_', '')

# Actual outcome
def actual_outcome(row):
    if row['home_score'] > row['away_score']: return 'home'
    if row['home_score'] < row['away_score']: return 'away'
    return 'draw'

valid['actual_outcome'] = valid.apply(actual_outcome, axis=1)

# Accuracy
accuracy = (valid['pred_outcome'] == valid['actual_outcome']).mean()
print(f"Outcome accuracy: {accuracy:.1%}")
print(f"\nConfusion:")
print(pd.crosstab(valid['actual_outcome'], valid['pred_outcome']))

- Outcome accuracy: 60.5% — solid, in the range of credible football models (538, bookmakers' implied probabilities are typically 55-62%).
- That number answered: "How often was the highest-probability outcome the actual outcome?"
- The confusion matrix: the column for "draw" predictions is missing entirely because the model never picks draw as the most likely outcome (it's almost always the second-place option). All 432 actual draws got misclassified — 277 as home wins, 155 as away wins.
- For home the model gets 750 out of 1027 (73% GREAT)
- For Away the model gets 372 out of 641  (58% good)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Home wins: what P(home win) did the model assign?
home_wins = valid[valid['actual_outcome'] == 'home']['p_home']
axes[0].hist(home_wins, bins=20, color='green', alpha=0.7, edgecolor='black')
axes[0].axvline(home_wins.mean(), color='red', linestyle='--',
                label=f'Mean = {home_wins.mean():.2f}')
axes[0].set_title(f'Actual home wins ({len(home_wins)} matches)')
axes[0].set_xlabel('Model\'s P(home win)')
axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_xlim(0, 1)

# Draws: what P(draw) did the model assign?
draws = valid[valid['actual_outcome'] == 'draw']['p_draw']
axes[1].hist(draws, bins=20, color='gray', alpha=0.7, edgecolor='black')
axes[1].axvline(draws.mean(), color='red', linestyle='--',
                label=f'Mean = {draws.mean():.2f}')
axes[1].set_title(f'Actual draws ({len(draws)} matches)')
axes[1].set_xlabel('Model\'s P(draw)')
axes[1].set_ylabel('Count')
axes[1].legend()
axes[1].set_xlim(0, 1)

# Away wins
away_wins = valid[valid['actual_outcome'] == 'away']['p_away']
axes[2].hist(away_wins, bins=20, color='orange', alpha=0.7, edgecolor='black')
axes[2].axvline(away_wins.mean(), color='red', linestyle='--',
                label=f'Mean = {away_wins.mean():.2f}')
axes[2].set_title(f'Actual away wins ({len(away_wins)} matches)')
axes[2].set_xlabel('Model\'s P(away win)')
axes[2].set_ylabel('Count')
axes[2].legend()
axes[2].set_xlim(0, 1)

plt.suptitle('What probability did the model assign to the actual outcome?', fontsize=13)
plt.tight_layout()
plt.show()

The chart answers a different question: "When a specific outcome actually happened, what probability did the model assign to it?"
The three means come from three different subsets of matches:

- 0.65 — restricted to the 864 matches where home actually won. Across those matches, the model assigned an average P(home win) of 0.65.
- 0.23 — restricted to the 432 matches where a draw actually happened. Across those, the model's average P(draw) was 0.23.
- 0.50 — restricted to the 558 matches where away actually won. Across those, the model's average P(away win) was 0.50.

#### the chart means are more useful for the project.

## Check 3 - Calibration plot

In [ ]:
import matplotlib.pyplot as plt

# Bin predictions by their probability
bins = np.linspace(0, 1, 11)  # 0-10%, 10-20%, ..., 90-100%
valid['p_home_bin'] = pd.cut(valid['p_home'], bins=bins)

# For each bin, compute actual home win rate
calibration = valid.groupby('p_home_bin', observed=True).agg(
    predicted=('p_home', 'mean'),
    actual=('actual_outcome', lambda x: (x == 'home').mean()),
    n=('p_home', 'size')
).reset_index()

print(calibration)

# Plot
plt.figure(figsize=(7, 7))
plt.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
plt.scatter(calibration['predicted'], calibration['actual'],
            s=calibration['n']*0.5, alpha=0.7, label='Model')
plt.xlabel('Predicted P(home win)')
plt.ylabel('Actual home win rate')
plt.title('Calibration plot — home win predictions')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

- This is genuinely good calibration. Every dot is within ~5 percentage points of the diagonal.

# 
----

# SAVE Models 

## Save trained models

Persist both Poisson models and the feature column order to `../models/` so the simulation stage can load them.

In [ ]:
import pickle
from pathlib import Path

MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

with open(MODELS_DIR / "poisson_home.pkl", "wb") as f:
    pickle.dump(model_home, f)

with open(MODELS_DIR / "poisson_away.pkl", "wb") as f:
    pickle.dump(model_away, f)

# Save feature column order so simulation can align columns identically
with open(MODELS_DIR / "feature_columns.pkl", "wb") as f:
    pickle.dump(X_train.columns.tolist(), f)

print("✓ Models saved to ../models/")
print("  poisson_home.pkl")
print("  poisson_away.pkl")
print("  feature_columns.pkl")


---
# Section 6 — Tournament Simulation (Monte Carlo)

Use the trained Poisson models to simulate the full 2026 World Cup many times.

**Pipeline:**
1. Load trained models and lookup tables (Elo, form, H2H, confederation, FIFA rank)
2. Build a match prediction function for any two teams
3. Simulate group-stage matches and produce group standings
4. Select 32 qualified teams (top 2 per group + best 8 third-placed)
5. Build and simulate the knockout bracket (R32 → R16 → QF → SF → Final)
6. Repeat 1,000 times (Monte Carlo) to estimate stage-reach probabilities


In [ ]:
import pandas as pd
import numpy as np
import pickle
from pathlib import Path

# Section 1: Load models and data

In [ ]:
# Project paths
DATA_DIR = Path("../data")
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REFERENCE_DIR = DATA_DIR / "reference"
MODELS_DIR = Path("../models")

In [ ]:
# Load trained Poisson models
with open(MODELS_DIR / "poisson_home.pkl", "rb") as f:
    model_home = pickle.load(f)

with open(MODELS_DIR / "poisson_away.pkl", "rb") as f:
    model_away = pickle.load(f)

# Load feature column order used during training
with open(MODELS_DIR / "feature_columns.pkl", "rb") as f:
    feature_columns = pickle.load(f)

print("Models loaded successfully.")
print(f"Number of model features: {len(feature_columns)}")

In [ ]:
# Core datasets
df_historical = pd.read_csv(INTERIM_DIR / "historical_matches.csv")
df_fixtures = pd.read_csv(INTERIM_DIR / "wc2026_fixtures.csv")

# Processed feature datasets
df_match_features = pd.read_csv(PROCESSED_DIR / "df_match_features.csv")
df_form = pd.read_csv(PROCESSED_DIR / "df_form_2026.csv")
df_h2h = pd.read_csv(PROCESSED_DIR / "df_h2h_2026.csv")

# Reference datasets
df_confederations = pd.read_csv(REFERENCE_DIR / "FIFA_confederations.csv")
df_knockout = pd.read_csv(REFERENCE_DIR / "fixtures_knockout_wc2026.csv")
df_groups = pd.read_csv(REFERENCE_DIR / "group_stages.csv", sep=";")

# FIFA ranking reference
df_fifa_rank = pd.read_csv(DATA_DIR / "raw" / "wc_2026_48_teams_fifa_rank_change_corrected.csv")

print("Datasets loaded successfully.")

In [ ]:
# Inspect shape
datasets = {
    "historical_matches": df_historical,
    "wc2026_fixtures": df_fixtures,
    "df_match_features": df_match_features,
    "df_form_2026": df_form,
    "df_h2h_2026": df_h2h,
    "FIFA_confederations": df_confederations,
    "group_stages": df_groups,
    "fixtures_knockout_wc2026": df_knockout,
    "fifa_rank_2026": df_fifa_rank,
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

In [ ]:
# Inspect columns
for name, df in datasets.items():
    print(f"\n{name}")
    print(df.columns.tolist())

In [ ]:
# Check number of World Cup teams
teams_from_groups = set(df_groups["nation"])
teams_from_confederations = set(df_confederations["nation"])
teams_from_form = set(df_form["team"])
teams_from_fifa = set(df_fifa_rank["Nation"])

print("Teams in group_stages:", len(teams_from_groups))
print("Teams in confederations:", len(teams_from_confederations))
print("Teams in form table:", len(teams_from_form))
print("Teams in FIFA ranking table:", len(teams_from_fifa))

# Check missing teams across key reference tables
print("\nTeams in groups but missing from confederations:")
print(sorted(teams_from_groups - teams_from_confederations))

print("\nTeams in groups but missing from form table:")
print(sorted(teams_from_groups - teams_from_form))

print("\nTeams in groups but missing from FIFA ranking table:")
print(sorted(teams_from_groups - teams_from_fifa))

In [ ]:
display(df_fixtures.head())
display(df_groups.head())
display(df_knockout.head())
display(df_match_features.head())

#
----

# Section 2: create lookup tables for Elo, form, H2H, and confederation.

The aim here is to make it easy to fetch, for any team:
- Elo rating
- confederation
- recent form score
- FIFA rank
- H2H score against another team

## 2.1— Create lookup tables

Before we can predict any 2026 match, we need quick lookup tables for team-level and matchup-level information.

In this section, we create:

- latest Elo rating per team
- team-to-confederation mapping
- team-to-form-score mapping
- team-to-FIFA-rank mapping
- head-to-head lookup between two teams

These lookup tables will make the prediction and simulation functions much cleaner.

In [ ]:
#### prepare latest Elo per team

# Make sure dates are treated as dates
df_match_features["date"] = pd.to_datetime(df_match_features["date"])

# Home-team Elo records
home_elo = df_match_features[["date", "home_team", "home_elo_pre"]].rename(
    columns={
        "home_team": "team",
        "home_elo_pre": "elo"
    }
)

# Away-team Elo records
away_elo = df_match_features[["date", "away_team", "away_elo_pre"]].rename(
    columns={
        "away_team": "team",
        "away_elo_pre": "elo"
    }
)

# Combine home and away Elo records
df_team_elo = pd.concat([home_elo, away_elo], ignore_index=True)

# Get latest available Elo before the 2026 tournament
df_latest_elo = (
    df_team_elo
    .sort_values("date")
    .drop_duplicates(subset="team", keep="last")
    .reset_index(drop=True)
)

team_to_elo = dict(zip(df_latest_elo["team"], df_latest_elo["elo"]))

df_latest_elo.tail()

In [ ]:
df_latest_elo.sort_values("elo", ascending=False).head(13)

In [ ]:
#### create confederation lookup

team_to_confederation = dict(
    zip(df_confederations["nation"], df_confederations["confederation"])
)

list(team_to_confederation.items())[:5]

In [ ]:
#### create form lookup (last 10 games goal differential)

team_to_form = dict(
    zip(df_form["team"], df_form["form_score"])
)

list(team_to_form.items())[:5]

In [ ]:
#### create FIFA ranking lookup as of 2026

team_to_fifa_rank = dict(
    zip(df_fifa_rank["Nation"], df_fifa_rank["FIFA_2026_rank"])
)

team_to_fifa_rank_change = dict(
    zip(df_fifa_rank["Nation"], df_fifa_rank["rank_change"])
)

list(team_to_fifa_rank.items())[:8]

In [ ]:
#### create H2H helper function

def get_h2h_score(team_a, team_b):
    """
    Return the head-to-head score from team_a's perspective.

    Example:
    If Argentina vs Brazil = +6,
    then Brazil vs Argentina = -6.
    """

    direct_match = df_h2h[
        (df_h2h["team_a"] == team_a) &
        (df_h2h["team_b"] == team_b)
    ]

    if len(direct_match) > 0:
        return direct_match["h2h_score"].iloc[0]

    reverse_match = df_h2h[
        (df_h2h["team_a"] == team_b) &
        (df_h2h["team_b"] == team_a)
    ]

    if len(reverse_match) > 0:
        return -reverse_match["h2h_score"].iloc[0]

    return 0

In [ ]:
#### test H2H helper (team head to head, only looks at last 5 games but gives 0 for teams that only played 1 or 2 games)

test_pairs = [
    ("Argentina", "Brazil"),
    ("Brazil", "Argentina"),
    ("France", "Spain"),
    ("Spain", "France"),
    ("France", "Senegal"),
    ("Algeria", "Morocco")
]

for team_a, team_b in test_pairs:
    print(f"{team_a} vs {team_b}: {get_h2h_score(team_a, team_b)}")

In [ ]:
#### sanity check all World Cup teams have lookup values

wc_teams = sorted(df_groups["nation"].unique())

missing_elo = [team for team in wc_teams if team not in team_to_elo]
missing_confederation = [team for team in wc_teams if team not in team_to_confederation]
missing_form = [team for team in wc_teams if team not in team_to_form]
missing_fifa_rank = [team for team in wc_teams if team not in team_to_fifa_rank]

print("Missing Elo:", missing_elo)
print("Missing confederation:", missing_confederation)
print("Missing form:", missing_form)
print("Missing FIFA rank:", missing_fifa_rank)

In [ ]:
#### inspect one team profile

sample_team = "Morocco"

team_profile = {
    "team": sample_team,
    "elo": team_to_elo.get(sample_team),
    "confederation": team_to_confederation.get(sample_team),
    "form_score": team_to_form.get(sample_team),
    "fifa_rank": team_to_fifa_rank.get(sample_team),
    "rank_change": team_to_fifa_rank_change.get(sample_team),
}

team_profile

#### Choice:
- Keep the prediction model pure. Use only the features it was trained and validated on. 
- Use FIFA rank, form, and H2H only for dashboard context and explanations.

#### Use the model for probabilities
Poisson model:
- Elo + match context → expected goals → win/draw/loss probabilities

#### Use the extra football context for interpretation:
Dashboard explanation:
- FIFA rank, rank change, recent form, H2H, confederation, Elo difference

PS: I also calculated recent form, FIFA rankings, and head-to-head records. But I did not feed them into the model because they were not part of the validated training setup. Instead, I use them in the dashboard to explain each matchup alongside the model probabilities

#
----

# Section 3 - Build match prediction system

## 3.1 — Build match prediction function

Now that the lookup tables are ready, we can create a prediction function for any 2026 matchup.

The model only uses the features it was trained on:

- home Elo
- away Elo
- Elo difference
- tournament weight
- neutral venue
- home confederation
- away confederation

The function will:

1. Build a one-row feature table
2. One-hot encode categorical columns
3. Align the columns with `feature_columns.pkl`
4. Predict expected goals using the two Poisson models
5. Convert expected goals into win/draw/loss probabilities

## Build Model feature row

In [ ]:
def build_match_features(home_team, away_team, neutral=True, tournament_weight=5):
    """
    Build one prediction row for a 2026 World Cup match.

    This function only includes the features used during model training.
    """

    home_elo = team_to_elo.get(home_team)
    away_elo = team_to_elo.get(away_team)

    home_conf = team_to_confederation.get(home_team)
    away_conf = team_to_confederation.get(away_team)

    if home_elo is None:
        raise ValueError(f"Missing Elo rating for {home_team}")

    if away_elo is None:
        raise ValueError(f"Missing Elo rating for {away_team}")

    if home_conf is None:
        raise ValueError(f"Missing confederation for {home_team}")

    if away_conf is None:
        raise ValueError(f"Missing confederation for {away_team}")

    row = pd.DataFrame([{
        "home_elo_pre": home_elo,
        "away_elo_pre": away_elo,
        "elo_diff": home_elo - away_elo,
        "tournament_weight": tournament_weight,
        "neutral": int(neutral),
        "home_confederation": home_conf,
        "away_confederation": away_conf
    }])

    # One-hot encode categorical variables
    X = pd.get_dummies(row)

    # Align with training columns
    X = X.reindex(columns=feature_columns, fill_value=0)

    # Important: statsmodels intercept column must be 1
    if "const" in X.columns:
        X["const"] = 1

    # Make sure all model inputs are numeric
    X = X.astype(float)

    return X

- It creates the exact one-row input table that the trained Poisson model expects.

- It is not predicting yet. It is only preparing the features.

In [ ]:
#### test feature builder

X_test = build_match_features("Argentina", "Brazil")

print(X_test.shape)
display(X_test)

## create Poisson probability function

In [ ]:
from scipy.stats import poisson

In [ ]:
_predict_match_cache = {}

def predict_match(home_team, away_team, neutral=True, tournament_weight=5, max_goals=10):
    """
    Predict expected goals and win/draw/loss probabilities for one match.
    Cached to speed up Monte Carlo simulations.
    """
    cache_key = (home_team, away_team, neutral, tournament_weight, max_goals)
    if cache_key in _predict_match_cache:
        return _predict_match_cache[cache_key]

    X = build_match_features(
        home_team=home_team,
        away_team=away_team,
        neutral=neutral,
        tournament_weight=tournament_weight
    )

    home_xg = model_home.predict(X)[0]
    away_xg = model_away.predict(X)[0]

    score_probs = []

    home_win_prob = 0
    draw_prob = 0
    away_win_prob = 0

    for home_goals in range(max_goals + 1):
        for away_goals in range(max_goals + 1):

            prob = (
                poisson.pmf(home_goals, home_xg) *
                poisson.pmf(away_goals, away_xg)
            )

            score_probs.append({
                "home_goals": home_goals,
                "away_goals": away_goals,
                "probability": prob
            })

            if home_goals > away_goals:
                home_win_prob += prob
            elif home_goals == away_goals:
                draw_prob += prob
            else:
                away_win_prob += prob

    score_probs = pd.DataFrame(score_probs)

    result = {
        "home_team": home_team,
        "away_team": away_team,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "home_win_prob": home_win_prob,
        "draw_prob": draw_prob,
        "away_win_prob": away_win_prob,
        "score_probs": score_probs
    }
    _predict_match_cache[cache_key] = result
    return result

## Test Predictions

In [ ]:
test_matches = [
    ("Argentina", "Brazil"),
    ("France", "Spain"),
    ("Morocco", "Portugal"),
    ("England", "Iran"),
    ("Mexico", "South Africa")
]

for home_team, away_team in test_matches:
    pred = predict_match(home_team, away_team)

    print(f"\n{home_team} vs {away_team}")
    print(f"Home xG: {pred['home_xg']:.2f}")
    print(f"Away xG: {pred['away_xg']:.2f}")
    print(f"Home win: {pred['home_win_prob']:.1%}")
    print(f"Draw: {pred['draw_prob']:.1%}")
    print(f"Away win: {pred['away_win_prob']:.1%}")

In [ ]:
df_latest_elo.sort_values("elo", ascending=False).head(3)

In [ ]:
#### test H2H helper (team head to head, only looks at last 5 games but gives 0 for teams that only played 1 or 2 games)

test_pairs = [
    ("France", "Spain"),
    ("Portugal", "Morocco")
]

for team_a, team_b in test_pairs:
    print(f"{team_a} vs {team_b}: {get_h2h_score(team_a, team_b)}")

## inspect most likely scorelines

In [ ]:
pred = predict_match("Argentina", "Brazil")

pred["score_probs"].sort_values(
    "probability",
    ascending=False
).head(10)

In [ ]:
pred = predict_match("Portugal", "Morocco")

pred["score_probs"].sort_values(
    "probability",
    ascending=False
).head(10)

In [ ]:
pred["home_win_prob"] + pred["draw_prob"] + pred["away_win_prob"]

- The win/draw/loss probabilities sum to almost 1.
- The most likely scorelines are realistic for the predicted expected goals.
- This suggests the prediction function is working as expected.

#
-----

# Section 4 — simulate a single match.

## 4.1 — Simulate a single match

Now that we can predict expected goals for any matchup, we can simulate one match result.

The logic is:

1. Use the Poisson model to predict expected goals for each team
2. Sample actual goals from each team's Poisson distribution
3. Return the score and match result

For group-stage matches, draws are allowed.
Knockout matches will need extra logic later because one team must advance.

In [ ]:
def simulate_match(home_team, away_team, neutral=True, tournament_weight=5, random_state=None):
    """
    Simulate one football match using the predicted expected goals.

    For group-stage matches:
    - home win is allowed
    - draw is allowed
    - away win is allowed
    """

    if random_state is not None:
        np.random.seed(random_state)

    pred = predict_match(
        home_team=home_team,
        away_team=away_team,
        neutral=neutral,
        tournament_weight=tournament_weight
    )

    home_xg = pred["home_xg"]
    away_xg = pred["away_xg"]

    home_goals = np.random.poisson(home_xg)
    away_goals = np.random.poisson(away_xg)

    if home_goals > away_goals:
        result = "H"
        winner = home_team
    elif home_goals < away_goals:
        result = "A"
        winner = away_team
    else:
        result = "D"
        winner = None

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "home_goals": home_goals,
        "away_goals": away_goals,
        "result": result,
        "winner": winner
    }

In [ ]:
#### Test one match

simulate_match("Argentina", "Brazil", random_state=16)

In [ ]:
simulate_match("Argentina", "Brazil", random_state=42)

- One simulated match can look surprising. That is normal.

- The whole point of Monte Carlo is that we do not trust one run. We run the tournament thousands of times, then count the patterns.

## Simulate the same match multiple times

In [ ]:
for i in range(10):
    match = simulate_match("Argentina", "Brazil")

    print(
        f"{match['home_team']} {match['home_goals']} - "
        f"{match['away_goals']} {match['away_team']} | "
        f"Result: {match['result']}"
    )

## Test a few different match-ups

In [ ]:
sample_matches = [
    ("Argentina", "Brazil"),
    ("France", "Spain"),
    ("Morocco", "Portugal"),
    ("England", "Iran"),
    ("Mexico", "South Africa")
]

for home_team, away_team in sample_matches:
    match = simulate_match(home_team, away_team)

    print(
        f"{match['home_team']} {match['home_goals']} - "
        f"{match['away_goals']} {match['away_team']} "
        f"| xG: {match['home_xg']:.2f} - {match['away_xg']:.2f} "
        f"| Result: {match['result']}"
    )

In [ ]:
### Simulate 10,000 times to see if averages make sense

results = []

for i in range(10_000):
    match = simulate_match("Argentina", "Brazil")
    results.append(match["result"])

pd.Series(results).value_counts(normalize=True).sort_index()

The single-match outputs will look noisy. That is the whole point of simulation. What matters is that thousands of simulated matches average out to the model probabilities.

So now we have confirmed three things:

1. predict_match() gives sensible xG and probabilities.
2. simulate_match() creates random match outcomes correctly.
3. Repeating the same match many times converges back to the model probabilities.

- ##### One small note: 10_000 took 42 seconds, which is slow because each simulation calls predict_match() again. Later, for the full tournament, we should avoid recalculating xG every time where possible.

# 
-----

# Section 5 — simulate the group stage.

## 5.1 — Simulate the group stage

Now we simulate the 72 group-stage matches.

For each match, we will:

1. Simulate the score using `simulate_match()`
2. Award points
3. Update goals for, goals against, and goal difference
4. Build one table per group
5. Rank teams using points, goal difference, and goals scored

For this first version, if teams are still tied after those rules, we use a random tie-breaker.

## add group information to fixtures

In [ ]:
# Team to group lookup
team_to_group = dict(zip(df_groups["nation"], df_groups["group"]))

# Add group to each group-stage fixture
df_group_fixtures = df_fixtures.copy()

df_group_fixtures["group"] = df_group_fixtures["home_team"].map(team_to_group)

# Basic check
display(df_group_fixtures.head())
print(df_group_fixtures["group"].value_counts().sort_index())

Why 6 games per group:
- Team 1 vs Team 2
- Team 1 vs Team 3
- Team 1 vs Team 4
- Team 2 vs Team 3
- Team 2 vs Team 4
- Team 3 vs Team 4

In [ ]:
#### check every fixture has a group

missing_group_fixtures = df_group_fixtures[df_group_fixtures["group"].isna()]

print("Fixtures with missing group:", len(missing_group_fixtures))
display(missing_group_fixtures)

## create empty group table

In [ ]:
#### create empty group table

def create_empty_group_table(group_teams):
    """
    Create an empty standings table for one group.
    """

    table = pd.DataFrame({
        "team": group_teams,
        "played": 0,
        "wins": 0,
        "draws": 0,
        "losses": 0,
        "goals_for": 0,
        "goals_against": 0,
        "goal_difference": 0,
        "points": 0
    })

    return table

## Update table after 1 Match

In [ ]:
def update_group_table(table, match):
    """
    Update a group table after one simulated match.
    """

    home_team = match["home_team"]
    away_team = match["away_team"]
    home_goals = match["home_goals"]
    away_goals = match["away_goals"]

    # Update played
    table.loc[table["team"] == home_team, "played"] += 1
    table.loc[table["team"] == away_team, "played"] += 1

    # Update goals
    table.loc[table["team"] == home_team, "goals_for"] += home_goals
    table.loc[table["team"] == home_team, "goals_against"] += away_goals

    table.loc[table["team"] == away_team, "goals_for"] += away_goals
    table.loc[table["team"] == away_team, "goals_against"] += home_goals

    # Update result stats and points
    if home_goals > away_goals:
        table.loc[table["team"] == home_team, "wins"] += 1
        table.loc[table["team"] == away_team, "losses"] += 1

        table.loc[table["team"] == home_team, "points"] += 3

    elif home_goals < away_goals:
        table.loc[table["team"] == away_team, "wins"] += 1
        table.loc[table["team"] == home_team, "losses"] += 1

        table.loc[table["team"] == away_team, "points"] += 3

    else:
        table.loc[table["team"] == home_team, "draws"] += 1
        table.loc[table["team"] == away_team, "draws"] += 1

        table.loc[table["team"] == home_team, "points"] += 1
        table.loc[table["team"] == away_team, "points"] += 1

    # Recalculate goal difference
    table["goal_difference"] = table["goals_for"] - table["goals_against"]

    return table

## rank one group table

In [ ]:
def rank_group_table(table):
    """
    Rank teams in a group.

    Main rules:
    1. Points
    2. Goal difference
    3. Goals scored
    4. Random tie-breaker
    """

    table = table.copy()

    # Temporary random tie-breaker for exact ties
    table["random_tiebreaker"] = np.random.random(len(table))

    table = (
        table
        .sort_values(
            by=["points", "goal_difference", "goals_for", "random_tiebreaker"],
            ascending=[False, False, False, False]
        )
        .reset_index(drop=True)
    )

    table["group_rank"] = table.index + 1

    table = table.drop(columns=["random_tiebreaker"])

    return table

## simulate one full group

In [ ]:
def simulate_group(group_name):
    """
    Simulate all matches in one group and return the ranked group table.
    """

    group_teams = (
        df_groups[df_groups["group"] == group_name]
        .sort_values("position")["nation"]
        .tolist()
    )

    group_matches = df_group_fixtures[df_group_fixtures["group"] == group_name]

    table = create_empty_group_table(group_teams)

    simulated_matches = []

    for _, row in group_matches.iterrows():
        match = simulate_match(
            home_team=row["home_team"],
            away_team=row["away_team"],
            neutral=row["neutral"]
        )

        simulated_matches.append(match)
        table = update_group_table(table, match)

    ranked_table = rank_group_table(table)
    ranked_table["group"] = group_name

    return ranked_table, pd.DataFrame(simulated_matches)

## test one group

In [ ]:
group_a_table, group_a_matches = simulate_group("C")

display(group_a_matches)
display(group_a_table)

## Run 1000 iterations on 1 group 

In [ ]:
group_c_rank_results = []

for i in range(1000):
    group_table, group_matches = simulate_group("C")

    group_c_rank_results.append(
        group_table[["team", "group_rank"]]
    )

df_group_c_ranks = pd.concat(group_c_rank_results, ignore_index=True)

group_c_rank_probs = (
    df_group_c_ranks
    .groupby(["team", "group_rank"])
    .size()
    .reset_index(name="count")
)

group_c_rank_probs["probability"] = group_c_rank_probs["count"] / 1000

group_c_rank_probs_pivot = (
    group_c_rank_probs
    .pivot(index="team", columns="group_rank", values="probability")
    .fillna(0)
)

group_c_rank_probs_pivot.columns = [
    f"rank_{int(col)}_prob" for col in group_c_rank_probs_pivot.columns
]

group_c_rank_probs_pivot.sort_values("rank_1_prob", ascending=False)

## Group stage full 12 Group simulation

In [ ]:
def simulate_group_stage():
    """
    Simulate all 12 groups and return:
    - full ranked group tables
    - all simulated group-stage matches
    """

    all_group_tables = []
    all_group_matches = []

    for group_name in sorted(df_groups["group"].unique()):
        group_table, group_matches = simulate_group(group_name)

        all_group_tables.append(group_table)
        all_group_matches.append(group_matches)

    df_group_tables = pd.concat(all_group_tables, ignore_index=True)
    df_group_matches = pd.concat(all_group_matches, ignore_index=True)

    return df_group_tables, df_group_matches

In [ ]:
#### only creates one random tournament group-stage sample 

df_group_tables, df_group_matches = simulate_group_stage()

display(df_group_matches.head())
display(df_group_tables.head())

print("Group-stage matches:", df_group_matches.shape)
print("Group tables:", df_group_tables.shape)

In [ ]:
#### Sanity check

print("Number of simulated group matches:", len(df_group_matches))
print("Number of teams in group tables:", len(df_group_tables))

print("\nPlayed matches per team:")
print(df_group_tables["played"].value_counts())

print("\nTeams per group:")
print(df_group_tables["group"].value_counts().sort_index())

#
----

# Section 6 — Select qualified teams

## 6.1 — Select qualified teams

After the group stage, 32 teams qualify for the knockout stage.

The 2026 format is:

- Top 2 from each group qualify automatically
- Best 8 third-placed teams also qualify
- Total: 32 teams

In this section, we create:

- direct qualifiers: group winners and runners-up
- best third-placed teams
- a slot mapping for group winners and runners-up, such as `1A`, `2A`, `1B`, `2B`

## get direct qualifiers

In [ ]:
def get_direct_qualifiers(df_group_tables):
    """
    Select group winners and runners-up.

    Returns a dictionary like:
    {
        "1A": "Mexico",
        "2A": "South Korea",
        ...
    }
    """

    direct_qualifiers = {}

    top_two = df_group_tables[df_group_tables["group_rank"].isin([1, 2])].copy()

    for _, row in top_two.iterrows():
        slot = f"{int(row['group_rank'])}{row['group']}"
        direct_qualifiers[slot] = row["team"]

    return direct_qualifiers

In [ ]:
df_group_tables, df_group_matches = simulate_group_stage()

display(df_group_tables.head())
print(df_group_tables.shape)

In [ ]:
#### test direct qualifiers

direct_qualifiers = get_direct_qualifiers(df_group_tables)

print("Number of direct qualifiers:", len(direct_qualifiers))
direct_qualifiers

## get best third-placed teams

In [ ]:
def get_best_third_placed_teams(df_group_tables, n_teams=8):
    """
    Select the best third-placed teams.

    Ranking rules used:
    1. Points
    2. Goal difference
    3. Goals scored
    4. FIFA 2026 rank as final practical tie-breaker

    Note:
    Official rules may use fair play before FIFA ranking,
    but we do not simulate yellow/red cards in this project.
    """

    third_placed = df_group_tables[df_group_tables["group_rank"] == 3].copy()

    third_placed["fifa_rank"] = third_placed["team"].map(team_to_fifa_rank)

    best_third_placed = (
        third_placed
        .sort_values(
            by=["points", "goal_difference", "goals_for", "fifa_rank"],
            ascending=[False, False, False, True]
        )
        .head(n_teams)
        .reset_index(drop=True)
    )

    best_third_placed["third_place_rank"] = best_third_placed.index + 1

    return best_third_placed

In [ ]:
best_third_placed = get_best_third_placed_teams(df_group_tables)

display(best_third_placed)

print("Number of best third-placed teams:", len(best_third_placed))

## combine qualifiers

In [ ]:
def get_qualified_teams(df_group_tables):
    """
    Get all 32 teams that qualify for the knockout stage.

    Returns:
    - direct_qualifiers: dictionary for 1A, 2A, 1B, 2B...
    - best_third_placed: dataframe of the 8 best third-placed teams
    - qualified_teams: list of all 32 qualified teams
    """

    direct_qualifiers = get_direct_qualifiers(df_group_tables)
    best_third_placed = get_best_third_placed_teams(df_group_tables)

    direct_teams = list(direct_qualifiers.values())
    third_placed_teams = best_third_placed["team"].tolist()

    qualified_teams = direct_teams + third_placed_teams

    return direct_qualifiers, best_third_placed, qualified_teams

## test all qualified teams

In [ ]:
direct_qualifiers, best_third_placed, qualified_teams = get_qualified_teams(df_group_tables)

print("Direct qualifiers:", len(direct_qualifiers))
print("Best third-placed teams:", len(best_third_placed))
print("Total qualified teams:", len(qualified_teams))
print("Unique qualified teams:", len(set(qualified_teams)))

display(best_third_placed)

## create third-place group lookup

This will help in the next section when we deal with bracket slots like:
- 3ABCDF
- 3CDFGH
- 3CEFHI

In [ ]:
third_place_team_to_group = dict(
    zip(best_third_placed["team"], best_third_placed["group"])
)

third_place_group_to_team = dict(
    zip(best_third_placed["group"], best_third_placed["team"])
)

print("Qualified third-place teams by group:")
third_place_group_to_team

In [ ]:
#### Final sanity check function

def check_qualification_output(direct_qualifiers, best_third_placed, qualified_teams):
    """
    Run sanity checks for knockout qualification.
    """

    print("Direct qualifier slots:", len(direct_qualifiers))
    print("Best third-placed teams:", len(best_third_placed))
    print("Total qualified teams:", len(qualified_teams))
    print("Unique qualified teams:", len(set(qualified_teams)))

    if len(direct_qualifiers) != 24:
        print("Issue: expected 24 direct qualifier slots.")

    if len(best_third_placed) != 8:
        print("Issue: expected 8 best third-placed teams.")

    if len(qualified_teams) != 32:
        print("Issue: expected 32 qualified teams.")

    if len(set(qualified_teams)) != 32:
        print("Issue: duplicate qualified teams found.")

    print("\nDirect qualifier slots:")
    for slot, team in sorted(direct_qualifiers.items()):
        print(f"{slot}: {team}")

    print("\nBest third-placed teams:")
    display(best_third_placed[[
        "team",
        "group",
        "points",
        "goal_difference",
        "goals_for",
        "third_place_rank"
    ]])

In [ ]:
check_qualification_output(
    direct_qualifiers=direct_qualifiers,
    best_third_placed=best_third_placed,
    qualified_teams=qualified_teams
)

#
------

# Section 7 - Build match prediction

## 7.1 — Build the Round of 32 bracket

Now that we know the 32 qualified teams, we need to fill the knockout bracket.

The bracket contains slot codes such as:

- `1A`: winner of Group A
- `2B`: runner-up of Group B
- `3ABCDF`: a third-placed team from one of those groups

For direct slots like `1A` or `2B`, the mapping is simple.

For third-place slots, we assign the best available third-placed team whose group is allowed by the slot code.

## inspect knockout bracket file

In [ ]:
display(df_knockout.head())
print(df_knockout.columns.tolist())
print(df_knockout.shape)

## find Round of 32 rows

In [ ]:
df_knockout["round"].value_counts()

In [ ]:
df_round_32 = df_knockout[df_knockout["round"] == "R32"].copy()

display(df_round_32)
print("Round of 32 matches:", len(df_round_32))

## create third-place resolver

- This function resolves slots like: 3ABCDF

In [ ]:
from itertools import permutations

def assign_third_place_slots(third_place_slots, best_third_placed):
    """
    Assign third-placed teams to all third-place bracket slots.

    This tries possible assignments and keeps the first valid one.

    A valid assignment means:
    - each third-place slot gets one team
    - the team's group is allowed by that slot
    - no team is used twice
    """

    third_teams = best_third_placed[["team", "group", "third_place_rank"]].copy()

    if len(third_place_slots) != len(third_teams):
        raise ValueError(
            f"Number of third-place slots ({len(third_place_slots)}) does not match "
            f"number of third-place teams ({len(third_teams)})."
        )

    # Try every possible order of the 8 third-placed teams
    for perm in permutations(third_teams.to_dict("records")):
        assignment = {}
        valid = True

        for slot, team_row in zip(third_place_slots, perm):
            allowed_groups = list(str(slot).replace("3", ""))

            if team_row["group"] not in allowed_groups:
                valid = False
                break

            assignment[slot] = team_row["team"]

        if valid:
            return assignment

    raise ValueError("No valid third-place assignment found.")

## build Round of 32 fixtures

In [ ]:
direct_qualifiers, best_third_placed, qualified_teams = get_qualified_teams(df_group_tables)

In [ ]:
def build_round_32_bracket(df_round_32, direct_qualifiers, best_third_placed):
    """
    Fill the Round of 32 bracket with real teams.

    Handles:
    - direct slots like 1A, 2B
    - third-place slots like 3ABCDF
    """

    df_round_32 = df_round_32.copy()

    # Find all unique third-place slots in the R32 bracket
    third_place_slots = []

    for col in ["home_slot", "away_slot"]:
        slots = (
            df_round_32[col]
            .astype(str)
            .loc[lambda s: s.str.startswith("3")]
            .unique()
            .tolist()
        )
        third_place_slots.extend(slots)

    third_place_slots = list(dict.fromkeys(third_place_slots))

    # Assign third-placed teams globally, not one-by-one
    third_place_assignment = assign_third_place_slots(
        third_place_slots=third_place_slots,
        best_third_placed=best_third_placed
    )

    filled_matches = []

    for _, row in df_round_32.iterrows():

        home_slot = str(row["home_slot"])
        away_slot = str(row["away_slot"])

        if home_slot in direct_qualifiers:
            home_team = direct_qualifiers[home_slot]
        elif home_slot.startswith("3"):
            home_team = third_place_assignment[home_slot]
        else:
            raise ValueError(f"Unsupported home slot: {home_slot}")

        if away_slot in direct_qualifiers:
            away_team = direct_qualifiers[away_slot]
        elif away_slot.startswith("3"):
            away_team = third_place_assignment[away_slot]
        else:
            raise ValueError(f"Unsupported away slot: {away_slot}")

        match = row.to_dict()
        match["home_team"] = home_team
        match["away_team"] = away_team

        filled_matches.append(match)

    df_round_32_filled = pd.DataFrame(filled_matches)

    return df_round_32_filled

In [ ]:
df_round_32_filled = build_round_32_bracket(
    df_round_32=df_round_32,
    direct_qualifiers=direct_qualifiers,
    best_third_placed=best_third_placed
)

display(df_round_32_filled)

- Note: Third-place teams are assigned using the allowed group letters in the bracket slot. 
- If several qualified third-place teams fit the same slot, the best-ranked available third-place team is selected. 
- This is a practical simplification of the full official placement matrix.

In [ ]:
#### Sanity checks

round_32_teams = (
    df_round_32_filled["home_team"].tolist() +
    df_round_32_filled["away_team"].tolist()
)

print("Round of 32 matches:", len(df_round_32_filled))
print("Teams in Round of 32:", len(round_32_teams))
print("Unique teams in Round of 32:", len(set(round_32_teams)))

duplicates = pd.Series(round_32_teams).value_counts()
duplicates = duplicates[duplicates > 1]

print("\nDuplicate teams:")
display(duplicates)

missing_from_qualified = set(qualified_teams) - set(round_32_teams)

print("\nQualified teams missing from Round of 32:")
print(sorted(missing_from_qualified))

#
-----

# Section 8 — Simulate knockout matches and propagate winners.

## 8.1 — Simulate knockout matches

The knockout stage is different from the group stage because every match must have a winner.

The logic is:

1. Simulate the 90-minute score using the Poisson model
2. If one team wins, that team advances
3. If the match is drawn, choose the advancing team using normalized model win probabilities
4. Send the winner into the next match using the bracket file

## simulate one knockout match

In [ ]:
def simulate_knockout_match(home_team, away_team, neutral=True, tournament_weight=5):
    """
    Simulate one knockout match.

    If the 90-minute result is a draw, choose the advancing team
    using normalized model win probabilities.
    """

    pred = predict_match(
        home_team=home_team,
        away_team=away_team,
        neutral=neutral,
        tournament_weight=tournament_weight
    )

    home_xg = pred["home_xg"]
    away_xg = pred["away_xg"]

    home_goals = np.random.poisson(home_xg)
    away_goals = np.random.poisson(away_xg)

    # Normal-time winner
    if home_goals > away_goals:
        winner = home_team
        loser = away_team
        result_type = "normal_time"

    elif away_goals > home_goals:
        winner = away_team
        loser = home_team
        result_type = "normal_time"

    else:
        # If draw, choose who advances based on model win strength
        home_strength = pred["home_win_prob"]
        away_strength = pred["away_win_prob"]

        home_advance_prob = home_strength / (home_strength + away_strength)

        if np.random.random() < home_advance_prob:
            winner = home_team
            loser = away_team
        else:
            winner = away_team
            loser = home_team

        result_type = "draw_resolved"

    return {
        "home_team": home_team,
        "away_team": away_team,
        "home_xg": home_xg,
        "away_xg": away_xg,
        "home_goals": home_goals,
        "away_goals": away_goals,
        "winner": winner,
        "loser": loser,
        "result_type": result_type,
        "home_win_prob": pred["home_win_prob"],
        "draw_prob": pred["draw_prob"],
        "away_win_prob": pred["away_win_prob"]
    }

## test one knockout match

In [ ]:
knockout_test = simulate_knockout_match("Brazil", "Morocco")

knockout_test

## simulate one full knockout round

In [ ]:
def simulate_knockout_round(df_round_matches):
    """
    Simulate one knockout round.

    Returns:
    - simulated match results
    - winners mapping by match_id
    """

    simulated_results = []
    winners_by_match = {}

    for _, row in df_round_matches.iterrows():

        match_result = simulate_knockout_match(
            home_team=row["home_team"],
            away_team=row["away_team"]
        )

        match_result["match_id"] = row["match_id"]
        match_result["round"] = row["round"]
        match_result["winner_advances_to"] = row["winner_advances_to"]
        match_result["loser_advances_to"] = row["loser_advances_to"]

        simulated_results.append(match_result)

        winners_by_match[row["match_id"]] = match_result["winner"]

    df_results = pd.DataFrame(simulated_results)

    return df_results, winners_by_match

## test Round of 32 simulation

In [ ]:
df_r32_results, r32_winners = simulate_knockout_round(df_round_32_filled)

display(df_r32_results)

print("Number of R32 matches:", len(df_r32_results))
print("Number of R32 winners:", len(r32_winners))

## fill next round from previous winners

In [ ]:
def build_next_round(df_knockout, previous_round_results, next_round_name):
    """
    Build the next knockout round using winners from the previous round.

    Example:
    R32 winners fill R16 matches.
    """

    df_next_round = df_knockout[df_knockout["round"] == next_round_name].copy()

    # Match ID -> winner
    winner_to_next_match = dict(
        zip(
            previous_round_results["match_id"],
            previous_round_results["winner"]
        )
    )

    # Next match -> list of teams that should appear there
    next_match_teams = {}

    for _, row in previous_round_results.iterrows():
        next_match_id = row["winner_advances_to"]
        winner = row["winner"]

        if pd.isna(next_match_id):
            continue

        if next_match_id not in next_match_teams:
            next_match_teams[next_match_id] = []

        next_match_teams[next_match_id].append(winner)

    filled_matches = []

    for _, row in df_next_round.iterrows():
        match_id = row["match_id"]

        teams = next_match_teams.get(match_id, [])

        if len(teams) != 2:
            raise ValueError(
                f"Expected 2 teams for match {match_id}, got {len(teams)}: {teams}"
            )

        match = row.to_dict()
        match["home_team"] = teams[0]
        match["away_team"] = teams[1]

        filled_matches.append(match)

    return pd.DataFrame(filled_matches)

## build and simulate Round of 16

In [ ]:
df_round_16_filled = build_next_round(
    df_knockout=df_knockout,
    previous_round_results=df_r32_results,
    next_round_name="R16"
)

display(df_round_16_filled)

df_r16_results, r16_winners = simulate_knockout_round(df_round_16_filled)

display(df_r16_results)

print("Number of R16 matches:", len(df_r16_results))
print("Number of R16 winners:", len(r16_winners))

## build and simulate QF, SF, Final

In [ ]:
# Quarter-finals
df_qf_filled = build_next_round(
    df_knockout=df_knockout,
    previous_round_results=df_r16_results,
    next_round_name="QF"
)

df_qf_results, qf_winners = simulate_knockout_round(df_qf_filled)

# Semi-finals
df_sf_filled = build_next_round(
    df_knockout=df_knockout,
    previous_round_results=df_qf_results,
    next_round_name="SF"
)

df_sf_results, sf_winners = simulate_knockout_round(df_sf_filled)

# Final
df_final_filled = build_next_round(
    df_knockout=df_knockout,
    previous_round_results=df_sf_results,
    next_round_name="Final"
)

df_final_results, final_winners = simulate_knockout_round(df_final_filled)

display(df_qf_results)
display(df_sf_results)
display(df_final_results)

winner = df_final_results["winner"].iloc[0]
runner_up = df_final_results["loser"].iloc[0]

print("World Cup winner:", winner)
print("Runner-up:", runner_up)

## combine all knockout results

In [ ]:
df_knockout_results = pd.concat(
    [
        df_r32_results,
        df_r16_results,
        df_qf_results,
        df_sf_results,
        df_final_results
    ],
    ignore_index=True
)

display(df_knockout_results)

print("Total knockout matches simulated:", len(df_knockout_results))

- R32: 16
- R16: 8
- QF: 4
- SF: 2
- Final: 1
- Total: 31

## create one full knockout function

In [ ]:
def simulate_knockout_stage(df_knockout, df_round_32_filled):
    """
    Simulate the full knockout stage from R32 to Final.

    Returns:
    - all knockout results
    - winner
    - runner_up
    """

    # R32
    df_r32_results, _ = simulate_knockout_round(df_round_32_filled)

    # R16
    df_round_16_filled = build_next_round(
        df_knockout=df_knockout,
        previous_round_results=df_r32_results,
        next_round_name="R16"
    )
    df_r16_results, _ = simulate_knockout_round(df_round_16_filled)

    # QF
    df_qf_filled = build_next_round(
        df_knockout=df_knockout,
        previous_round_results=df_r16_results,
        next_round_name="QF"
    )
    df_qf_results, _ = simulate_knockout_round(df_qf_filled)

    # SF
    df_sf_filled = build_next_round(
        df_knockout=df_knockout,
        previous_round_results=df_qf_results,
        next_round_name="SF"
    )
    df_sf_results, _ = simulate_knockout_round(df_sf_filled)

    # Final
    df_final_filled = build_next_round(
        df_knockout=df_knockout,
        previous_round_results=df_sf_results,
        next_round_name="Final"
    )
    df_final_results, _ = simulate_knockout_round(df_final_filled)

    df_knockout_results = pd.concat(
        [
            df_r32_results,
            df_r16_results,
            df_qf_results,
            df_sf_results,
            df_final_results
        ],
        ignore_index=True
    )

    winner = df_final_results["winner"].iloc[0]
    runner_up = df_final_results["loser"].iloc[0]

    return df_knockout_results, winner, runner_up

## test full knockout stage

In [ ]:
df_knockout_results, winner, runner_up = simulate_knockout_stage(
    df_knockout=df_knockout,
    df_round_32_filled=df_round_32_filled
)

display(df_knockout_results)

print("Winner:", winner)
print("Runner-up:", runner_up)
print("Matches:", len(df_knockout_results))

#
----

# Section 9 - Full tournament simulation

## 9.1 — Simulate one full tournament

Now that the group-stage and knockout-stage logic work separately, we combine them into one full tournament simulation.

One tournament simulation will:

1. Simulate all 72 group-stage matches
2. Rank all 12 group tables
3. Select the 32 qualified teams
4. Build the Round of 32 bracket
5. Simulate the knockout stage
6. Return the winner, runner-up, and all stage results

This still represents only one possible tournament outcome.
The Monte Carlo section will repeat this many times to estimate probabilities.

## Simulate tournament function

In [ ]:
def simulate_tournament():
    """
    Simulate one full World Cup tournament.

    Returns:
    - tournament_summary: dictionary with winner, runner-up, and stage teams
    - df_group_tables: final group-stage tables
    - df_group_matches: simulated group-stage matches
    - df_round_32_filled: filled Round of 32 bracket
    - df_knockout_results: all knockout match results
    """

    # 1. Simulate group stage
    df_group_tables, df_group_matches = simulate_group_stage()

    # 2. Select qualified teams
    direct_qualifiers, best_third_placed, qualified_teams = get_qualified_teams(
        df_group_tables
    )

    # 3. Build Round of 32 bracket
    df_round_32_filled = build_round_32_bracket(
        df_round_32=df_round_32,
        direct_qualifiers=direct_qualifiers,
        best_third_placed=best_third_placed
    )

    # 4. Simulate knockout stage
    df_knockout_results, winner, runner_up = simulate_knockout_stage(
        df_knockout=df_knockout,
        df_round_32_filled=df_round_32_filled
    )

    # 5. Collect stage teams
    r32_teams = qualified_teams

    r16_teams = (
        df_knockout_results
        .loc[df_knockout_results["round"] == "R32", "winner"]
        .tolist()
    )

    qf_teams = (
        df_knockout_results
        .loc[df_knockout_results["round"] == "R16", "winner"]
        .tolist()
    )

    sf_teams = (
        df_knockout_results
        .loc[df_knockout_results["round"] == "QF", "winner"]
        .tolist()
    )

    final_teams = (
        df_knockout_results
        .loc[df_knockout_results["round"] == "SF", "winner"]
        .tolist()
    )

    tournament_summary = {
        "winner": winner,
        "runner_up": runner_up,
        "r32_teams": r32_teams,
        "r16_teams": r16_teams,
        "qf_teams": qf_teams,
        "sf_teams": sf_teams,
        "final_teams": final_teams
    }

    return {
        "summary": tournament_summary,
        "group_tables": df_group_tables,
        "group_matches": df_group_matches,
        "round_32_bracket": df_round_32_filled,
        "knockout_results": df_knockout_results
    }

## test one full tournament

In [ ]:
tournament = simulate_tournament()

tournament["summary"]

## inspect group-stage output

In [ ]:
display(tournament["group_matches"].head(6))
display(tournament["group_tables"].head(8))

## inspect Round of 32 bracket

In [ ]:
display(tournament["round_32_bracket"])

## inspect knockout results

In [ ]:
display(tournament["knockout_results"])

print("Knockout matches:", len(tournament["knockout_results"]))
print("Winner:", tournament["summary"]["winner"])
print("Runner-up:", tournament["summary"]["runner_up"])

## full tournament sanity check

In [ ]:
summary = tournament["summary"]

print("R32 teams:", len(summary["r32_teams"]))
print("R16 teams:", len(summary["r16_teams"]))
print("QF teams:", len(summary["qf_teams"]))
print("SF teams:", len(summary["sf_teams"]))
print("Final teams:", len(summary["final_teams"]))
print("Winner:", summary["winner"])
print("Runner-up:", summary["runner_up"])

print("\nUnique R32 teams:", len(set(summary["r32_teams"])))
print("Unique R16 teams:", len(set(summary["r16_teams"])))
print("Unique QF teams:", len(set(summary["qf_teams"])))
print("Unique SF teams:", len(set(summary["sf_teams"])))
print("Unique Final teams:", len(set(summary["final_teams"])))

## optional nicer summary

In [ ]:
print("One simulated World Cup outcome")
print("-" * 35)
print(f"Winner: {summary['winner']}")
print(f"Runner-up: {summary['runner_up']}")
print(f"Semi-finalists: {summary['sf_teams']}")
print(f"Finalists: {summary['final_teams']}")

#
----

## Section 10 — Monte Carlo simulation

## 10.1 — Run Monte Carlo simulations

Now that one full tournament simulation works, we can repeat it many times.

Each simulation creates one possible World Cup outcome.

By running the tournament hundreds or thousands of times, we estimate:

- probability of reaching the Round of 32
- probability of reaching the Round of 16
- probability of reaching the quarter-finals
- probability of reaching the semi-finals
- probability of reaching the final
- probability of winning the tournament

This is the main output of the project.

## create Monte Carlo function

In [ ]:
def run_monte_carlo_simulations(n_simulations=100):
    """
    Run the full World Cup simulation many times.

    Returns:
    - df_simulation_results: one row per team with stage counts and probabilities
    - all_winners: list of winners from each simulation
    """

    wc_teams = sorted(df_groups["nation"].unique())

    stage_counts = {
        team: {
            "r32_count": 0,
            "r16_count": 0,
            "qf_count": 0,
            "sf_count": 0,
            "final_count": 0,
            "winner_count": 0
        }
        for team in wc_teams
    }

    all_winners = []

    for i in range(n_simulations):
        tournament = simulate_tournament()
        summary = tournament["summary"]

        for team in summary["r32_teams"]:
            stage_counts[team]["r32_count"] += 1

        for team in summary["r16_teams"]:
            stage_counts[team]["r16_count"] += 1

        for team in summary["qf_teams"]:
            stage_counts[team]["qf_count"] += 1

        for team in summary["sf_teams"]:
            stage_counts[team]["sf_count"] += 1

        for team in summary["final_teams"]:
            stage_counts[team]["final_count"] += 1

        winner = summary["winner"]
        stage_counts[winner]["winner_count"] += 1
        all_winners.append(winner)

        if (i + 1) % 50 == 0:
            print(f"Completed {i + 1}/{n_simulations} simulations")

    df_simulation_results = pd.DataFrame.from_dict(
        stage_counts,
        orient="index"
    ).reset_index().rename(columns={"index": "team"})

    # Convert counts to probabilities
    probability_columns = {
        "r32_count": "r32_prob",
        "r16_count": "r16_prob",
        "qf_count": "qf_prob",
        "sf_count": "sf_prob",
        "final_count": "final_prob",
        "winner_count": "winner_prob"
    }

    for count_col, prob_col in probability_columns.items():
        df_simulation_results[prob_col] = (
            df_simulation_results[count_col] / n_simulations
        )

    df_simulation_results = df_simulation_results.sort_values(
        "winner_prob",
        ascending=False
    ).reset_index(drop=True)

    return df_simulation_results, all_winners

## test with 10 simulations first

In [ ]:
df_simulation_results_10, winners_10 = run_monte_carlo_simulations(
    n_simulations=10
)

display(df_simulation_results_10.head(10))
pd.Series(winners_10).value_counts()

## Run 100 simulations

In [ ]:
df_simulation_results_100, winners_100 = run_monte_carlo_simulations(
    n_simulations=100
)

display(df_simulation_results_100.head(15))

## inspect winner probabilities

In [ ]:
df_simulation_results_100[
    [
        "team",
        "r32_prob",
        "r16_prob",
        "qf_prob",
        "sf_prob",
        "final_prob",
        "winner_prob"
    ]
].head(20)

## sanity checks

In [ ]:
print("Teams:", len(df_simulation_results_100))

print("\nWinner probability sum:")
print(df_simulation_results_100["winner_prob"].sum())

print("\nFinal probability sum:")
print(df_simulation_results_100["final_prob"].sum())

print("\nSemi-final probability sum:")
print(df_simulation_results_100["sf_prob"].sum())

print("\nQuarter-final probability sum:")
print(df_simulation_results_100["qf_prob"].sum())

print("\nRound of 16 probability sum:")
print(df_simulation_results_100["r16_prob"].sum())

print("\nRound of 32 probability sum:")
print(df_simulation_results_100["r32_prob"].sum())

## run 1,000 simulations

In [ ]:
df_simulation_results_1000, winners_1000 = run_monte_carlo_simulations(
    n_simulations=1000 
)

display(
    df_simulation_results_1000[
        [
            "team",
            "r32_prob",
            "r16_prob",
            "qf_prob",
            "sf_prob",
            "final_prob",
            "winner_prob"
        ]
    ].head(20)
)

## add context columns for dashboard

In [ ]:
df_final_probs = df_simulation_results_1000.copy()

df_final_probs["elo"] = df_final_probs["team"].map(team_to_elo)
df_final_probs["confederation"] = df_final_probs["team"].map(team_to_confederation)
df_final_probs["form_score"] = df_final_probs["team"].map(team_to_form)
df_final_probs["fifa_rank"] = df_final_probs["team"].map(team_to_fifa_rank)
df_final_probs["rank_change"] = df_final_probs["team"].map(team_to_fifa_rank_change)

df_final_probs = df_final_probs[
    [
        "team",
        "confederation",
        "fifa_rank",
        "rank_change",
        "elo",
        "form_score",
        "r32_prob",
        "r16_prob",
        "qf_prob",
        "sf_prob",
        "final_prob",
        "winner_prob",
        "r32_count",
        "r16_count",
        "qf_count",
        "sf_count",
        "final_count",
        "winner_count"
    ]
]

df_final_probs.head(20)

In [ ]:
top_15 = df_final_probs.sort_values("winner_prob", ascending=False).head(15)

top_15.plot(
    x="team",
    y="winner_prob",
    kind="bar",
    figsize=(12, 5),
    legend=False,
    title="Top 15 World Cup 2026 winner probabilities"
)

## save final output

In [ ]:
OUTPUT_PATH = PROCESSED_DIR / "wc2026_tournament_probabilities.csv"

df_final_probs.to_csv(OUTPUT_PATH, index=False)

print(f"Saved final tournament probabilities to: {OUTPUT_PATH}")

In [ ]:
df_final_probs.groupby("confederation")[["r16_prob", "qf_prob", "sf_prob", "final_prob", "winner_prob"]].mean()

In [ ]:
df_final_probs.groupby("confederation")["winner_prob"].sum()